# Pipeline ETL — Dane energetyczne z ENTSO-E

Pobiera dane z [ENTSO-E Transparency Platform](https://transparency.entsoe.eu/) i zapisuje jako CSV.

Wymagania: klucz API w `.env` (`ENTSOE_API_KEY`). Plik surowy: `datasets/raw/energy_{KRAJ}_{OD}_{DO}.csv` (transformacja: `_data-transformation.ipynb`)

## 1. Konfiguracja

Dostępne kody krajów:

| Kod | Kraj | Kod | Kraj | Kod | Kraj |
|-----|------|-----|------|-----|------|
| AT | Austria | FR | Francja | NL | Holandia |
| BE | Belgia | DE | Niemcy | NO | Norwegia |
| BG | Bułgaria | GR | Grecja | PL | Polska |
| HR | Chorwacja | HU | Węgry | PT | Portugalia |
| CZ | Czechy | IE | Irlandia | RO | Rumunia |
| DK | Dania | IT | Włochy | RS | Serbia |
| EE | Estonia | LV | Łotwa | SK | Słowacja |
| FI | Finlandia | LT | Litwa | SI | Słowenia |
| ES | Hiszpania | LU | Luksemburg | SE | Szwecja |
| CH | Szwajcaria | GB | Wlk. Brytania | | |

In [34]:
COUNTRY_CODE = "BG"
START_DATE   = "2020-01-01"
END_DATE     = "2025-12-31"

## 2. Inicjalizacja

In [35]:
import os
import time
import warnings

import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient

warnings.filterwarnings("ignore")

# ── Mapowanie krajów na strefy czasowe ──
COUNTRY_TIMEZONES = {
    "AT": "Europe/Vienna",    "BE": "Europe/Brussels",  "BG": "Europe/Sofia",
    "HR": "Europe/Zagreb",    "CZ": "Europe/Prague",    "DK": "Europe/Copenhagen",
    "EE": "Europe/Tallinn",   "FI": "Europe/Helsinki",  "FR": "Europe/Paris",
    "DE": "Europe/Berlin",    "GR": "Europe/Athens",    "HU": "Europe/Budapest",
    "IE": "Europe/Dublin",    "IT": "Europe/Rome",      "LV": "Europe/Riga",
    "LT": "Europe/Vilnius",   "LU": "Europe/Luxembourg","NL": "Europe/Amsterdam",
    "NO": "Europe/Oslo",      "PL": "Europe/Warsaw",    "PT": "Europe/Lisbon",
    "RO": "Europe/Bucharest", "RS": "Europe/Belgrade",  "SK": "Europe/Bratislava",
    "SI": "Europe/Ljubljana", "ES": "Europe/Madrid",    "SE": "Europe/Stockholm",
    "CH": "Europe/Zurich",    "GB": "Europe/London",
}

if COUNTRY_CODE not in COUNTRY_TIMEZONES:
    raise ValueError(f"Nieznany kod kraju: {COUNTRY_CODE}. Dostępne: {', '.join(sorted(COUNTRY_TIMEZONES))}")

TIMEZONE = COUNTRY_TIMEZONES[COUNTRY_CODE]
OUTPUT_PATH = f"datasets/raw/energy_{COUNTRY_CODE}_{START_DATE}_{END_DATE}.csv"

print(f"Kraj:            {COUNTRY_CODE}")
print(f"Strefa czasowa:  {TIMEZONE} (automatycznie)")
print(f"Okres:           {START_DATE} → {END_DATE}")
print(f"Plik wyjściowy:  {OUTPUT_PATH}")

Kraj:            BG
Strefa czasowa:  Europe/Sofia (automatycznie)
Okres:           2020-01-01 → 2025-12-31
Plik wyjściowy:  datasets/raw/energy_BG_2020-01-01_2025-12-31.csv


In [36]:
load_dotenv()
API_KEY = os.getenv("ENTSOE_API_KEY")

if not API_KEY:
    raise ValueError("Brak klucza API!")

client = EntsoePandasClient(api_key=API_KEY)

start = pd.Timestamp(START_DATE, tz=TIMEZONE)
end   = pd.Timestamp(END_DATE, tz=TIMEZONE) + pd.Timedelta(days=1)

print(f"Api klient gotowy. Zakres zapytań: {start} → {end}")

Api klient gotowy. Zakres zapytań: 2020-01-01 00:00:00+02:00 → 2026-01-01 00:00:00+02:00


## 3. Funkcja pomocnicza — pobieranie danych w kawałkach

API ENTSO-E ogranicza ilość danych w jednym zapytaniu (maks. ~1 rok).  
Poniższa funkcja automatycznie dzieli zapytanie na mniejsze fragmenty i łączy wyniki.

In [37]:
def fetch_in_chunks(query_func, country_code, start, end, chunk_months=6, retries=3, **kwargs):
    chunks = []
    current = start

    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        print(f"  Pobieram: {current.date()} -> {chunk_end.date()} ... ")

        for attempt in range(1, retries + 1):
            try:
                result = query_func(country_code, start=current, end=chunk_end, **kwargs)
                chunks.append(result)
                print(f"OK ({len(result)} wierszy)")
                break
            except Exception as e:
                if attempt == retries:
                    print(e)
                    raise
                
                wait = 10 * attempt
                print(f"Błąd, ponawiam za {wait}s...")
                time.sleep(wait)

        current = chunk_end
        time.sleep(1)

    # Zabezpieczenie przed brakiem danych
    if not chunks:
        return None

    combined = pd.concat(chunks)
    return combined[~combined.index.duplicated(keep="first")].sort_index()


def safe_fetch(query_func, country_code, start, end, label, **kwargs):
    
    print(f"Pobieranie: {label}...\n")
    
    try:
        return fetch_in_chunks(query_func, country_code, start, end, **kwargs)
    except Exception as e:
        print(f"Brak danych (lub błąd) dla {country_code}: {e}")
        return None

## 4. Pobieranie danych

Pobieramy 4 typy danych z ENTSO-E:
- **Produkcja energii** (generation) — z podziałem na źródła
- **Prognozy** wiatru i słońca (forecasts)
- **Obciążenie sieci** — rzeczywiste i prognozowane (load)
- **Ceny energii** — rynek dnia następnego (prices)

Nie każdy kraj udostępnia wszystkie typy danych — brakujące zostaną oznaczone jako NaN.

In [38]:
generation_raw = safe_fetch(client.query_generation, COUNTRY_CODE, start, end,
                            "Produkcja energii (generation)")

forecast_raw = safe_fetch(client.query_wind_and_solar_forecast, COUNTRY_CODE, start, end,
                          "Prognozy wiatr/słońce (forecast)")

load_actual_raw = safe_fetch(client.query_load, COUNTRY_CODE, start, end,
                             "Obciążenie rzeczywiste (load actual)")

load_forecast_raw = safe_fetch(client.query_load_forecast, COUNTRY_CODE, start, end,
                               "Prognoza obciążenia (load forecast)")

price_da_raw = safe_fetch(client.query_day_ahead_prices, COUNTRY_CODE, start, end,
                          "Ceny day-ahead (prices)")

print("\nPobieranie zakończone.\n")

Pobieranie: Produkcja energii (generation)...

  Pobieram: 2020-01-01 -> 2020-07-01 ... 
OK (4367 wierszy)
  Pobieram: 2020-07-01 -> 2021-01-01 ... 
OK (4417 wierszy)
  Pobieram: 2021-01-01 -> 2021-07-01 ... 
OK (4342 wierszy)
  Pobieram: 2021-07-01 -> 2022-01-01 ... 
OK (4417 wierszy)
  Pobieram: 2022-01-01 -> 2022-07-01 ... 
OK (4342 wierszy)
  Pobieram: 2022-07-01 -> 2023-01-01 ... 
OK (4417 wierszy)
  Pobieram: 2023-01-01 -> 2023-07-01 ... 
OK (4343 wierszy)
  Pobieram: 2023-07-01 -> 2024-01-01 ... 
OK (4417 wierszy)
  Pobieram: 2024-01-01 -> 2024-07-01 ... 
OK (4367 wierszy)
  Pobieram: 2024-07-01 -> 2025-01-01 ... 
OK (4417 wierszy)
  Pobieram: 2025-01-01 -> 2025-07-01 ... 
OK (4343 wierszy)
  Pobieram: 2025-07-01 -> 2026-01-01 ... 
OK (4417 wierszy)
Pobieranie: Prognozy wiatr/słońce (forecast)...

  Pobieram: 2020-01-01 -> 2020-07-01 ... 
OK (4367 wierszy)
  Pobieram: 2020-07-01 -> 2021-01-01 ... 
OK (4368 wierszy)
  Pobieram: 2021-01-01 -> 2021-07-01 ... 
OK (4248 wierszy)
  Po

In [39]:
forecast_raw.head()

,Solar,Wind Onshore
2020-01-01 00:00:00+02:00,0.0,325.0
2020-01-01 01:00:00+02:00,0.0,362.0
2020-01-01 02:00:00+02:00,0.0,318.0
2020-01-01 03:00:00+02:00,0.0,370.0
2020-01-01 04:00:00+02:00,0.0,404.0


## 5. Łączenie surowych danych

Spłaszczenie kolumn z API i zapis bez transformacji (mapowanie i uzupełnianie braków → `_data-transformation.ipynb`).

In [40]:
def flatten_columns(raw_df):
    if raw_df is None:
        return pd.DataFrame()
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = [f"{a} - {b}" if b else a for a, b in raw_df.columns]
    return raw_df

def extract_series(raw, name):
    if raw is None:
        return pd.Series(dtype=float, name=name)
    if isinstance(raw, pd.DataFrame):
        s = raw.iloc[:, 0]
    else:
        s = raw
    s.name = name
    return s

generation = flatten_columns(generation_raw)
forecast = flatten_columns(forecast_raw)
load_actual = extract_series(load_actual_raw, "Actual Load")
load_forecast = extract_series(load_forecast_raw, "Forecasted Load")
price_da = extract_series(price_da_raw, "Price DA")

parts = [generation, forecast]
for s in [load_actual, price_da, load_forecast]:
    if len(s) > 0:
        parts.append(s.to_frame())

valid_parts = [p for p in parts if not p.empty]
df = pd.concat(valid_parts, axis=1)
df.index.name = "time"
df = df.sort_index().reset_index()

print(f"Połączono surowe dane! Kształt: {df.shape[0]} wierszy, {df.shape[1]} kolumn.")
print(f"Zakres dat: {df['time'].min()} -> {df['time'].max()}")

Połączono surowe dane! Kształt: 59233 wierszy, 17 kolumn.
Zakres dat: 2020-01-01 00:00:00+02:00 -> 2026-01-01 00:00:00+02:00


In [41]:
df.head()

,time,Biomass,Fossil Brown coal/Lignite,Fossil Gas,Fossil Hard coal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Nuclear,Solar,Waste,Wind Onshore,Solar,Wind Onshore,Actual Load,Price DA,Forecasted Load
0,2020-01-01 00:00:00+02:00,32.0,2098.0,400.0,57.0,202.0,46.0,9.0,2163.0,0.0,4.0,312.0,0.0,325.0,4410.0,54.20,4613.0
1,2020-01-01 01:00:00+02:00,31.0,1794.0,404.0,57.0,0.0,46.0,9.0,2163.0,0.0,4.0,340.0,0.0,362.0,4243.0,76.26,4444.0
2,2020-01-01 02:00:00+02:00,32.0,1722.0,405.0,58.0,0.0,48.0,9.0,2162.0,0.0,4.0,323.0,0.0,318.0,4063.0,68.57,4234.0
3,2020-01-01 03:00:00+02:00,32.0,1701.0,406.0,58.0,0.0,46.0,9.0,2163.0,0.0,4.0,306.0,0.0,370.0,3924.0,58.99,4064.0
4,2020-01-01 04:00:00+02:00,32.0,1643.0,406.0,58.0,0.0,47.0,9.0,2163.0,0.0,4.0,351.0,0.0,404.0,3838.0,52.51,3942.0


## 6. Zapis surowego CSV

In [42]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
print(f"Zapisano: {OUTPUT_PATH}")
print(f"Rozmiar:  {size_mb:.1f} MB")

Zapisano: datasets/raw/energy_BG_2020-01-01_2025-12-31.csv
Rozmiar:  6.4 MB


In [43]:
df.shape

(59233, 17)

In [44]:
df.describe()

,Biomass,Fossil Brown coal/Lignite,Fossil Gas,Fossil Hard coal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Nuclear,Solar,Waste,Wind Onshore,Solar,Wind Onshore,Actual Load,Price DA,Forecasted Load
count,52558.000000,52534.000000,52534.000000,52534.000000,52385.000000,52534.000000,52534.000000,52534.000000,52606.000000,52558.000000,52606.000000,51466.000000,52188.000000,52608.000000,59233.000000,52586.000000
mean,23.390582,1677.180719,260.873705,54.992825,16.991222,145.732063,236.974382,1821.875705,358.537580,3.414282,158.293050,376.258530,181.459556,4247.568914,141.390749,4254.024778
std,7.143182,792.689565,102.821135,15.279696,67.311763,90.799222,295.189333,444.212249,614.244366,1.502630,142.245276,671.915432,169.335476,877.172697,111.240019,887.259300
min,2.220000,262.500000,88.000000,0.000000,0.000000,10.020000,0.820000,410.000000,0.000000,0.000000,0.000000,0.000000,0.000000,30.670000,-100.630000,2473.000000
25%,18.000000,1041.720000,174.622500,50.110000,0.000000,69.982500,9.000000,1841.397500,0.010000,2.890000,46.000000,0.000000,42.000000,3638.995000,82.000000,3626.000000
50%,21.830000,1570.465000,237.609025,59.000000,0.000000,126.780987,99.490000,2028.380000,27.073052,3.240000,110.010000,4.000000,125.000000,4124.860000,108.440000,4122.000000
75%,30.000000,2245.000000,338.000000,63.517500,0.000000,207.756562,383.292500,2123.000000,490.000000,4.000000,236.000000,484.000000,284.000000,4796.975000,164.150000,4806.000000
max,122.670000,3624.050000,715.000000,157.000000,626.000000,422.000000,1577.000000,2173.000000,3888.800000,107.000000,737.710000,4103.000000,668.000000,7337.070000,1061.040000,7482.000000
